# 🩻 Foundation Models para Radiologia: ImageNet vs. Domínio Clínico
## Hands-on Vibe Coding — Sociedade Paulista de Radiologia 2026

---

### A Pergunta Central

Um modelo treinado em fotos do cotidiano (gatos, carros, 1.000 classes) tem o mesmo valor como extrator de características radiológicas que um modelo treinado em **880 mil raios-X de tórax**?

Neste hands-on você vai responder isso experimentalmente:

| Extrator | Pré-treinamento | O que aprendeu |
|---|---|---|
| **EfficientNetB0** | ImageNet (1,28M fotos naturais) | Bordas, texturas, formas gerais |
| **RAD-DINO** | 880K+ raios-X de tórax (Microsoft, 2024) | Opacidades, consolidações, textura pulmonar |

**Experimento:** congele ambos os backbones → treine a **mesma cabeça classificadora** em cada → compare o t-SNE dos embeddings e as métricas no conjunto de teste.

---

### Dataset: PneumoniaMNIST

- **1.000 imagens** de raio-X de tórax (64×64 pixels, subsample reproduzível)
- **2 classes:** Normal vs Pneumonia Bacteriana/Viral
- Download automático via `pip install medmnist` — sem login, sem cadastro
- Fonte: Kermany et al., *Cell* 2018 | Licença: CC BY 4.0

---

### Como usar este notebook

1. **Ative o GPU T4:** Menu `Ambiente de execução` → `Alterar tipo de hardware` → T4 GPU → Salvar → Reconectar
2. **Execute as células dos PC0, PC1 e PC2** — elas já estão pré-preenchidas e preparam todo o ambiente
3. A partir do **PC3**, use o **Gemini** (ícone ✦ ou `Ctrl+Shift+I`): copie o prompt do card projetado, cole no Gemini, cole o código gerado na célula e execute

> 💡 **PC0–PC2:** Execute diretamente (código pronto)  
> 🤖 **PC3–PC6:** Construa via Gemini (célula vazia aguardando seu código)

---

### ⏱️ Cronograma (75 minutos)

| | Bloco | Tempo | Modo |
|---|---|---|---|
| 🔧 | Setup e Preparação | 5 min | ▶ Execute |
| 📦 | Carregar e Visualizar o Dataset | 5 min | ▶ Execute |
| 🎬 | Extração de Embeddings + t-SNE | 10 min | ▶ Execute |
| 🏗️ | Classificador 1: EfficientNetB0 (ImageNet) | 12 min | ✦ Gemini |
| 🏗️ | Classificador 2: RAD-DINO (Radiologia) | 12 min | ✦ Gemini |
| 📊 | Comparação de Resultados | 8 min | ✦ Gemini |
| 🔎 | Inferência em Imagem Própria | 10 min | ✦ Gemini |
| 💬 | Discussão Clínica | 13 min | — |


## 🔧 Setup e Preparação
### Prompt Card 0 — ▶ Execute a célula abaixo


In [ ]:
import warnings
warnings.filterwarnings('ignore') # Ignorar avisos do medmnist

# ── Instalações ───────────────────────────────────────────────────────────────
!pip install -qqq medmnist torchvision transformers accelerate scikit-learn matplotlib seaborn

# ── Imports ───────────────────────────────────────────────────────────────────
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torchvision.transforms as transforms
import torchvision

from medmnist import PneumoniaMNIST

from transformers import AutoImageProcessor, AutoModel

from sklearn.metrics import accuracy_score, roc_auc_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.manifold import TSNE

# ── Setup Básico ──────────────────────────────────────────────────────────────
SEED = 42
def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Usando device: {DEVICE}')

# ── Funções Auxiliares ────────────────────────────────────────────────────────

def numpy_to_pil(images_np):
    """Converte um array numpy de imagens (N, H, W, C) para uma lista de imagens PIL."""
    images_np = (images_np * 255).astype(np.uint8)
    pil_images = [Image.fromarray(img[:,:,0] if img.shape[-1] == 1 else img) for img in images_np]
    return pil_images

def extrair_embeddings_cnn(model, images_np, batch_size=32):
    """Extrai embeddings de um modelo CNN (ex: EfficientNetB0)."""
    transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

    dataset = TensorDataset(torch.stack([transform(Image.fromarray((img * 255).astype(np.uint8))) for img in images_np]))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False)

    embeddings = []
    model.eval()
    with torch.no_grad():
        for batch in dataloader:
            img_batch = batch[0].to(DEVICE)
            emb = model(img_batch).cpu().numpy()
            embeddings.append(emb)
    return np.vstack(embeddings)

def extrair_embeddings_raddino(model, processor, images_pil, batch_size=32):
    """Extrai embeddings do modelo RAD-DINO."""
    embeddings = []
    model.eval()
    with torch.no_grad():
        for i in range(0, len(images_pil), batch_size):
            batch_pil = images_pil[i:i+batch_size]
            inputs = processor(images=batch_pil, return_tensors='pt').to(DEVICE)
            outputs = model(**inputs)
            emb = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.append(emb)
    return np.vstack(embeddings)

class ClassificationHead(nn.Module):
    def __init__(self, in_features, num_classes=2, dropout_rate=0.3):
        super().__init__()
        self.head = nn.Sequential(
            nn.Linear(in_features, 256),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        return self.head(x)

def treinar_cabeca(emb_train, y_train, emb_val, y_val, in_features, epochs=50, lr=1e-3, batch_size=32):
    """Treina a cabeça classificadora com os embeddings fornecidos."""
    set_seed(SEED)

    head = ClassificationHead(in_features).to(DEVICE)
    optimizer = torch.optim.Adam(head.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_dataset = TensorDataset(torch.tensor(emb_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
    val_dataset = TensorDataset(torch.tensor(emb_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

    history = {'loss': [], 'val_auc': []}

    for epoch in range(epochs):
        head.train()
        total_loss = 0
        for embeddings, labels in train_loader:
            embeddings, labels = embeddings.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            outputs = head(embeddings)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        history['loss'].append(total_loss / len(train_loader))

        head.eval()
        val_preds = []
        val_true = []
        val_probs = []
        with torch.no_grad():
            for embeddings, labels in val_loader:
                embeddings, labels = embeddings.to(DEVICE), labels.to(DEVICE)
                outputs = head(embeddings)
                val_probs.extend(torch.softmax(outputs, dim=1)[:, 1].cpu().numpy())
                val_preds.extend(torch.argmax(outputs, dim=1).cpu().numpy())
                val_true.extend(labels.cpu().numpy())
        val_auc = roc_auc_score(val_true, val_probs)
        history['val_auc'].append(val_auc)

    return head, history

def avaliar(head, emb_test, y_test, model_name):
    """Avalia o modelo e imprime métricas."""
    head.eval()
    with torch.no_grad():
        embeddings_tensor = torch.tensor(emb_test, dtype=torch.float32).to(DEVICE)
        outputs = head(embeddings_tensor)
        test_probs = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        test_preds = torch.argmax(outputs, dim=1).cpu().numpy()

    accuracy  = accuracy_score(y_test, test_preds)
    auc       = roc_auc_score(y_test, test_probs)
    precision = precision_score(y_test, test_preds)
    recall    = recall_score(y_test, test_preds)
    f1        = f1_score(y_test, test_preds)
    cm        = confusion_matrix(y_test, test_preds)

    print(f'\n--- Avaliação: {model_name} ---')
    print(f'Acurácia: {accuracy:.4f}')
    print(f'AUC: {auc:.4f}')
    print(f'Precisão: {precision:.4f}')
    print(f'Recall: {recall:.4f}')
    print(f'F1-Score: {f1:.4f}')
    print(f'Matriz de Confusão:\n{cm}')

    return {'model_name': model_name, 'accuracy': accuracy, 'auc': auc, 'precision': precision, 'recall': recall, 'f1': f1}

## 📦 Carregar e Visualizar o Dataset
### Prompt Card 1 — ▶ Execute a célula abaixo


In [ ]:
# Prompt Card 1: Carregar PneumoniaMNIST
train_ds = PneumoniaMNIST(split='train', download=True, size=64)
val_ds   = PneumoniaMNIST(split='val',   download=True, size=64)
test_ds  = PneumoniaMNIST(split='test',  download=True, size=64)

def preprocess(ds):
    imgs = ds.imgs.astype('float32') / 255.0
    if imgs.ndim == 3:
        imgs = imgs[..., np.newaxis]
    imgs   = np.repeat(imgs, 3, axis=-1)          # (N, 64, 64, 3)
    labels = ds.labels.flatten().astype('int')
    return imgs, labels

x_train, y_train = preprocess(train_ds)
x_val,   y_val   = preprocess(val_ds)
x_test,  y_test  = preprocess(test_ds)

# ── Subsample para 1000 imagens no total (~800 / 100 / 100) ───────────────────
rng_sub = np.random.default_rng(SEED)
for split, x_attr, y_attr, n in [
    ('x_train', x_train, y_train, 800),
    ('x_val',   x_val,   y_val,   100),
    ('x_test',  x_test,  y_test,  100),
]:
    idx = rng_sub.choice(len(x_attr), n, replace=False)
    if split == 'x_train': x_train, y_train = x_attr[idx], y_attr[idx]
    elif split == 'x_val': x_val,   y_val   = x_attr[idx], y_attr[idx]
    else:                  x_test,  y_test  = x_attr[idx], y_attr[idx]

class_names = ['Normal', 'Pneumonia']
for name, x, y in [('Treino', x_train, y_train), ('Validação', x_val, y_val), ('Teste', x_test, y_test)]:
    u, c = np.unique(y, return_counts=True)
    print(f'{name}: {x.shape} | {dict(zip([class_names[i] for i in u], c))}')

rng = np.random.default_rng(SEED)
idx = rng.choice(len(x_train), 16, replace=False)
fig, axes = plt.subplots(4, 4, figsize=(10, 10))
fig.suptitle('PneumoniaMNIST — Amostras do Treino (subsample 800)', fontsize=13, fontweight='bold')
for ax, i in zip(axes.flat, idx):
    ax.imshow(x_train[i, :, :, 0], cmap='gray')
    ax.set_title(class_names[y_train[i]], color='red' if y_train[i]==1 else 'green', fontsize=10)
    ax.axis('off')
plt.tight_layout()
plt.show()

## 🎬 Extração de Embeddings + t-SNE
### Prompt Card 2 — ▶ Execute a célula abaixo


In [ ]:
# Prompt Card 2: Extração de Embeddings + t-SNE

# 1. Carregar o EfficientNetB0 pré-treinado no ImageNet
print('⏳ Baixando EfficientNetB0 pré-treinado no ImageNet (~21MB)...')
backbone = torchvision.models.efficientnet_b0(weights='IMAGENET1K_V1')
for param in backbone.parameters():
    param.requires_grad = False

extrator_effnet = nn.Sequential(
    backbone.features,
    backbone.avgpool,
    nn.Flatten()
).to(DEVICE)
print('✅ EfficientNetB0 carregado e congelado!')

# 2. Extrair embeddings com EfficientNetB0 para todo o dataset (treino, validação e teste)
print('⏳ Extraindo embeddings com EfficientNetB0 para os datasets de treino, validação e teste...')
emb_train_effnet = extrair_embeddings_cnn(extrator_effnet, x_train)
emb_val_effnet   = extrair_embeddings_cnn(extrator_effnet, x_val)
emb_test_effnet  = extrair_embeddings_cnn(extrator_effnet, x_test)
print(f'   Shape dos embeddings do EfficientNetB0: Treino {emb_train_effnet.shape} | Val {emb_val_effnet.shape} | Teste {emb_test_effnet.shape}')

# 3. Carregar o RAD-DINO pré-treinado em radiologia
print('⏳ Baixando RAD-DINO da Microsoft (~87MB)...')
extrator_raddino  = AutoModel.from_pretrained('microsoft/rad-dino')
processor_raddino = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
for param in extrator_raddino.parameters():
    param.requires_grad = False
extrator_raddino = extrator_raddino.to(DEVICE)
print('✅ RAD-DINO carregado e congelado!')

# Converter imagens para formato PIL para o RAD-DINO (treino, validação e teste)
print('⏳ Convertendo imagens para formato PIL...')
imgs_pil_train = numpy_to_pil(x_train)
imgs_pil_val   = numpy_to_pil(x_val)
imgs_pil_test  = numpy_to_pil(x_test)
print('✅ Conversão concluída!')

# 4. Extrair embeddings com RAD-DINO para todo o dataset (treino, validação e teste)
print('⏳ Extraindo embeddings com RAD-DINO (CLS token, ~15s cada)...')
emb_train_raddino = extrair_embeddings_raddino(extrator_raddino, processor_raddino, imgs_pil_train)
emb_val_raddino   = extrair_embeddings_raddino(extrator_raddino, processor_raddino, imgs_pil_val)
emb_test_raddino  = extrair_embeddings_raddino(extrator_raddino, processor_raddino, imgs_pil_test)
print(f'   Shape dos embeddings do RAD-DINO: Treino {emb_train_raddino.shape} | Val {emb_val_raddino.shape} | Teste {emb_test_raddino.shape}')

# 5. Rodar t-SNE nos embeddings do conjunto de TESTE
print('⏳ Rodando t-SNE para visualizar os embeddings do conjunto de TESTE...')

tsne_effnet = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
emb_effnet_2d = tsne_effnet.fit_transform(emb_test_effnet)

tsne_raddino = TSNE(n_components=2, random_state=SEED, perplexity=30, n_iter=1000)
emb_raddino_2d = tsne_raddino.fit_transform(emb_test_raddino)

color_palette = {0: 'blue', 1: 'red'}

# 6. Plotar resultados do t-SNE
fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Visualização t-SNE dos Embeddings de Teste', fontsize=16, fontweight='bold')

from matplotlib.lines import Line2D
legend_elements = [
    Line2D([0], [0], marker='o', color='w', label='Normal', markerfacecolor='blue', markersize=8),
    Line2D([0], [0], marker='o', color='w', label='Pneumonia', markerfacecolor='red', markersize=8)
]

sns.scatterplot(x=emb_effnet_2d[:, 0], y=emb_effnet_2d[:, 1],
                hue=y_test, palette=color_palette, legend=False, ax=axes[0])
axes[0].set_title('EfficientNetB0 (ImageNet)')
axes[0].set_xlabel('t-SNE Componente 1')
axes[0].set_ylabel('t-SNE Componente 2')
axes[0].legend(handles=legend_elements, title='Classe')

sns.scatterplot(x=emb_raddino_2d[:, 0], y=emb_raddino_2d[:, 1],
                hue=y_test, palette=color_palette, legend=False, ax=axes[1])
axes[1].set_title('RAD-DINO (Radiologia)')
axes[1].set_xlabel('t-SNE Componente 1')
axes[1].set_ylabel('t-SNE Componente 2')
axes[1].legend(handles=legend_elements, title='Classe')

plt.tight_layout(rect=[0, 0.03, 1, 0.9])
plt.show()

## 🏗️ Classificador 1: EfficientNetB0 (ImageNet)
### Prompt Card 3 — ✦ Use o Gemini e cole o código na célula abaixo


In [ ]:
# ✦ Cole aqui o código gerado pelo Gemini para o Prompt Card 3


## 🏗️ Classificador 2: RAD-DINO (Radiologia)
### Prompt Card 4 — ✦ Use o Gemini e cole o código na célula abaixo


In [ ]:
# ✦ Cole aqui o código gerado pelo Gemini para o Prompt Card 4


## 📊 Comparação de Resultados
### Prompt Card 5 — ✦ Use o Gemini e cole o código na célula abaixo


In [ ]:
# ✦ Cole aqui o código gerado pelo Gemini para o Prompt Card 5


## 🔎 Inferência em Imagem Própria
### Prompt Card 6 — ✦ Use o Gemini e cole o código na célula abaixo


In [ ]:
# ✦ Cole aqui o código gerado pelo Gemini para o Prompt Card 6


---
---
# 💬 Discussão Clínica

## O que aconteceu?

Ambos os backbones ficaram **completamente congelados** durante o experimento. Apenas a cabeça classificadora foi treinada — e era idêntica para os dois:

| | EfficientNetB0 | RAD-DINO |
|---|---|---|
| **Pré-treinamento** | ImageNet (1,28M fotos naturais) | 880K+ RX de tórax (MIMIC, CheXpert, NIH, PadChest, BRAX) |
| **Arquitetura** | CNN (EfficientNet) | ViT-B/14 com DINOv2 |
| **Dim. embedding** | 1.280 | 768 |
| **Parâmetros treinados** | ~262K (só head) | ~262K (só head) |
| **Tamanho do modelo** | ~21MB | ~87MB |

## Perguntas para reflexão

**1. Por que o t-SNE do RAD-DINO mostra clusters mais separados?**
> Os embeddings do RAD-DINO já codificam padrões clínicos radiológicos (opacidades, consolidações, textura pulmonar anormal). O EfficientNetB0 ImageNet codifica bordas e texturas de imagens naturais — parcialmente útil, mas sem especificidade clínica.

**2. Por que treinamos APENAS a cabeça (feature extraction)?**
> Fine-tuning completo com 5.856 imagens em um backbone de 86M+ parâmetros causaria overfitting severo. Feature extraction é a abordagem padrão quando o dataset clínico é pequeno.

**3. O RAD-DINO foi treinado em pneumonia especificamente?**
> Não. Foi treinado de forma auto-supervisionada (DINOv2) em raios-X gerais — sem labels de diagnóstico. A separação que vemos vem do conhecimento geral de radiologia de tórax aprendido de forma não-supervisionada.

**4. Isso é suficiente para uso clínico real?**
> Não. Imagens 64×64 perdem detalhes diagnósticos críticos. O dataset tem viés geográfico (crianças de Guangzhou). Sem validação prospectiva nem aprovação regulatória. Este é um experimento educacional para demonstrar o princípio da **transferência de domínio**.

**5. Qual é a implicação prática para radiologia?**
> Ao escolher um modelo pré-treinado para fine-tuning em uma tarefa radiológica, modelos treinados em dados clínicos radiológicos tendem a superar modelos de propósito geral — mesmo quando apenas a cabeça é treinada.

---

## Referências

- **PneumoniaMNIST:** Kermany DS et al., *Cell*, 2018. doi: 10.1016/j.cell.2018.02.010
- **RAD-DINO:** Pérez-García F et al., *Nature Machine Intelligence*, 2025. doi: 10.1038/s42256-024-00965-w
- **DINOv2:** Oquab M et al., *TMLR*, 2024. arXiv: 2304.07193
- **MedMNIST v2:** Yang J et al., *Scientific Data*, 2023. doi: 10.1038/s41597-022-01721-8
- **EfficientNet:** Tan M, Le QV, *ICML*, 2019. arXiv: 1905.11946
